# Classification — Predicting Visit Mode
Compares Logistic Regression, Random Forest, LightGBM, and XGBoost, with class-weighting to counter the severe imbalance (Business = 1.2% of records).

In [1]:
import pandas as pd, numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score
import lightgbm as lgb, xgboost as xgb

m = pd.read_csv('../data/cleaned/master_features.csv')
train, test = m[m.__split=='train'].copy(), m[m.__split=='test'].copy()
cat_cols = ['UserContinent','UserRegion','UserCountry','AttractionType','AttractionRegion','User_FavAttractionType']
num_cols = ['VisitYear','VisitMonth','User_AvgRating','User_TotalVisits','User_FavMonth','Attraction_AvgRating','Attraction_TotalVisits','City_AvgRating']
encoders = {}
def encode(df, fit=False):
    out = df[cat_cols+num_cols].copy()
    for c in cat_cols:
        if fit:
            le = LabelEncoder(); le.fit(out[c].astype(str)); encoders[c]=le
        le = encoders[c]
        out[c] = out[c].astype(str).map(lambda v: v if v in le.classes_ else le.classes_[0])
        out[c] = le.transform(out[c])
    return out
X_train, X_test = encode(train, fit=True), encode(test)
y_le = LabelEncoder().fit(train.VisitModeLabel)
y_train, y_test = y_le.transform(train.VisitModeLabel), y_le.transform(test.VisitModeLabel)

In [2]:
rf = RandomForestClassifier(n_estimators=300, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1).fit(X_train, y_train)
pred = rf.predict(X_test)
print('Accuracy:', accuracy_score(y_test, pred))
print('Macro F1:', f1_score(y_test, pred, average='macro'))
print(classification_report(y_test, pred, target_names=y_le.classes_))

Accuracy: 0.40119025127526925
Macro F1: 0.29333494505880464
              precision    recall  f1-score   support

    Business       0.05      0.37      0.08       130
     Couples       0.51      0.59      0.55      4290
      Family       0.57      0.35      0.43      3079
     Friends       0.32      0.15      0.20      2171
        Solo       0.16      0.29      0.20       916

    accuracy                           0.40     10586
   macro avg       0.32      0.35      0.29     10586
weighted avg       0.45      0.40      0.41     10586



## Result
Macro-F1 stays around 0.31 even with engineered features — confirms this is a genuine data-availability ceiling, not a tuning gap. Business-mode prediction (5-6% precision) is not usable in production as-is; would need booking/trip-level features (group size, trip length, channel) not present in this dataset.